##Imports

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import *
from delta.tables import DeltaTable

##Configuration

In [0]:
CATALOG = "workspace"
SCHEMA = "final"

CONTROL_TABLE = f"{CATALOG}.{SCHEMA}.silver_load_control"

SOURCE_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer"

DEMO_TARGET_TABLE = f"{CATALOG}.{SCHEMA}.silver_customer_incremental_demo"

##Create Control Table

In [0]:
spark.sql(f"""

CREATE TABLE IF NOT EXISTS {CONTROL_TABLE}
(
    table_name STRING,
    last_load_timestamp TIMESTAMP,
    updated_timestamp TIMESTAMP
)

USING DELTA

""")

In [0]:
display(
    spark.table(CONTROL_TABLE)
)

##Seed Watermark Using Existing Silver Data

In [0]:
max_ts = (
    spark.table(SOURCE_TABLE)
    .agg(max("ingested_at"))
    .first()[0]
)

print(max_ts)

##Insert Initial Watermark

In [0]:
watermark_df = spark.createDataFrame(
    [
        (
            "silver_customer",
            max_ts
        )
    ],
    [
        "table_name",
        "last_load_timestamp"
    ]
).withColumn(
    "updated_timestamp",
    current_timestamp()
)

watermark_df.write \
    .mode("append") \
    .saveAsTable(CONTROL_TABLE)

In [0]:
display(
    spark.table(CONTROL_TABLE)
)

##Watermark Function

In [0]:
def get_last_watermark(table_name):

    result = (
        spark.table(CONTROL_TABLE)
        .filter(
            col("table_name") == table_name
        )
        .agg(
            max("last_load_timestamp")
        )
        .first()[0]
    )

    return result

In [0]:
watermark = get_last_watermark(
    "silver_customer"
)

print(watermark)

##Simulate New Incoming Data

In [0]:
customer_df = spark.table(
    SOURCE_TABLE
)

new_data_df = (
    customer_df
    .limit(5)
    .withColumn(
        "CustomerID",
        col("CustomerID") + 999999
    )
    .withColumn(
        "ingested_at",
        current_timestamp()
    )
)

In [0]:
display(new_data_df)

##Create Demo Source

In [0]:
source_demo_df = (
    customer_df
    .unionByName(new_data_df)
)

##Incremental Extraction

In [0]:
watermark = get_last_watermark(
    "silver_customer"
)

incremental_df = (
    source_demo_df
    .filter(
        col("ingested_at") > watermark
    )
)

In [0]:
print(
    f"Incremental Count : {incremental_df.count()}"
)

display(incremental_df)

##Create Demo Target Table

In [0]:
spark.sql(f"""
DROP TABLE IF EXISTS {DEMO_TARGET_TABLE}
""")

In [0]:
customer_df.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable(DEMO_TARGET_TABLE)

In [0]:
print(
    spark.table(
        DEMO_TARGET_TABLE
    ).count()
)

##Generic Merge Function

In [0]:
def merge_to_delta(
    source_df,
    target_table,
    merge_condition
):

    target = DeltaTable.forName(
        spark,
        target_table
    )

    (
        target.alias("t")
        .merge(
            source_df.alias("s"),
            merge_condition
        )
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

##Execute Merge

In [0]:
merge_to_delta(
    incremental_df,
    DEMO_TARGET_TABLE,
    "t.CustomerID = s.CustomerID"
)

In [0]:
print(
    spark.table(
        DEMO_TARGET_TABLE
    ).count()
)

##CDC Demonstration

In [0]:
customer_df = spark.table(
    SOURCE_TABLE
)

In [0]:
cdc_df = (
    customer_df
    .limit(1)
    .withColumn(
        "AccountNumber",
        lit("UPDATED_ACCOUNT")
    )
)

##Generate New Hash

In [0]:
cdc_df = (
    cdc_df
    .withColumn(
        "new_hash",
        sha2(
            concat_ws(
                "|",
                col("CustomerID").cast("string"),
                col("AccountNumber")
            ),
            256
        )
    )
)

##Existing Hash

In [0]:
existing_df = (
    customer_df
    .limit(1)
    .withColumn(
        "old_hash",
        sha2(
            concat_ws(
                "|",
                col("CustomerID").cast("string"),
                col("AccountNumber")
            ),
            256
        )
    )
)

##Compare Hashes

In [0]:
comparison_df = (
    existing_df
    .join(
        cdc_df.select(
            "CustomerID",
            "new_hash"
        ),
        "CustomerID"
    )
)

In [0]:
display(
    comparison_df.select(
        "CustomerID",
        "old_hash",
        "new_hash"
    )
)

##CDC Detection

In [0]:
cdc_changes = comparison_df.filter(
    col("old_hash") != col("new_hash")
)

In [0]:
display(cdc_changes)

##Update Watermark

In [0]:
latest_ts = (
    incremental_df
    .agg(max("ingested_at"))
    .first()[0]
)

In [0]:
spark.sql(f"""
DELETE FROM {CONTROL_TABLE}
WHERE table_name='silver_customer'
""")

In [0]:
spark.createDataFrame(
    [
        (
            "silver_customer",
            latest_ts
        )
    ],
    [
        "table_name",
        "last_load_timestamp"
    ]
).withColumn(
    "updated_timestamp",
    current_timestamp()
).write.mode("append") \
 .saveAsTable(CONTROL_TABLE)

##Final Verification

In [0]:
display(
    spark.table(CONTROL_TABLE)
)